In [23]:
import pandas as pd

In [24]:
INDEX_COLS = ['dv_rating_alpha', 'dv_rating_beta',
       'dv_rating_gamma', 'human_rating_alpha', 'human_rating_beta',
       'human_rating_gamma']

CODE_COLS = {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title', 'OEWS Code',
       'OEWS Title', 'CPS Code', 'CPS Title', 'manual_soc_to_cps'}

In [25]:
# Import Eloundou scores and finalized crosswalk
df_raw = pd.read_csv('../data/occ_level.csv')
df_raw= df_raw.drop(columns=['Title'])
crosswalk = pd.read_csv('../data/crosswalks/final_crosswalk.csv')

# Load OEWS permanent aggregate weights and employment total
oews_weights = pd.read_html("../data/crosswalks/Mapping Employment Projections and O_NET data_ a methodological overview _ Monthly Labor Review _ U.S. Bureau of Labor Statistics.html")[2]
oews = pd.read_excel('../data/national_M2024_dl.xlsx', sheet_name='national_M2024_dl', dtype=str)

In [26]:
# Merge scores
df = df_raw.merge(crosswalk,
                  left_on='O*NET-SOC Code',
                  right_on='ONET Code',
                  how='left',
                  indicator=True)

assert (df["_merge"] == "left_only").sum() == 0, "Some O*NET codes failed to merge"

df = df.drop(columns=['O*NET-SOC Code', '_merge'])

In [27]:
# Collapse scores down to the 6 digit level
df['agg_level_ONET'] = df.duplicated(subset=['SOC Code'], keep=False)

# Create standard deviation for merged codes
std_df_onet =  df.groupby('SOC Code')[INDEX_COLS].std().reset_index()
std_df_onet = std_df_onet.rename(columns={col: f'{col}_std_ONET' for col in INDEX_COLS})

# Merge in standard deviation and indicator of aggregation at this level
df_onet_with_stats = df.merge(std_df_onet, on='SOC Code', how='left')

collapsed_rows_onet = df_onet_with_stats[df_onet_with_stats['agg_level_ONET'] == True]
collapsed_rows_onet.to_csv('../temp/collapsed_at_O*NET.csv', index=False)

code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title'}

# Average the index values and keep all other columns
agg_dict = {col: 'mean' for col in INDEX_COLS}
agg_dict.update({col: 'first' for col in code_cols_for_agg})
agg_dict.update({f'{col}_std_ONET': 'first' for col in INDEX_COLS})
agg_dict.update({'agg_level_ONET': 'first'})

df_soc = df_onet_with_stats.groupby('SOC Code', as_index=False).agg(agg_dict)

assert len(df_soc) <= df['SOC Code'].nunique()
assert df_soc['SOC Code'].is_unique, "duplicate SOC Codes survived the groupby"

df_soc.to_csv('../temp/soc_level.csv', index=False)


In [28]:
# Testing difference in score means before and after aggregation

print(f"{'Column':<30} {'Mean (before)':>15} {'Mean (after)':>15} {'Diff':>10}")
print("-" * 72)

for col in INDEX_COLS:
    mean_before = df[col].mean()
    mean_after = df_soc[col].mean()
    diff = mean_after - mean_before
    print(f"{col:<30} {mean_before:>15.4f} {mean_after:>15.4f} {diff:>10.4f}")

# Remove top 2 variance inducing occupational codes

temp_df = df[~df['SOC Code'].isin(['15-1211', '15-1299'])]
temp_df_soc = df_soc[~df_soc['SOC Code'].isin(['15-1211', '15-1299'])]

print("-" * 72)

for col in INDEX_COLS:
    mean_before = temp_df[col].mean()
    mean_after = temp_df_soc[col].mean()
    diff = mean_after - mean_before
    print(f"{col:<30} {mean_before:>15.4f} {mean_after:>15.4f} {diff:>10.4f}")

Column                           Mean (before)    Mean (after)       Diff
------------------------------------------------------------------------
dv_rating_alpha                         0.1418          0.1379    -0.0039
dv_rating_beta                          0.3447          0.3257    -0.0190
dv_rating_gamma                         0.5477          0.5135    -0.0341
human_rating_alpha                      0.1441          0.1355    -0.0086
human_rating_beta                       0.3027          0.2857    -0.0170
human_rating_gamma                      0.4612          0.4358    -0.0254
------------------------------------------------------------------------
dv_rating_alpha                         0.1382          0.1373    -0.0009
dv_rating_beta                          0.3403          0.3248    -0.0154
dv_rating_gamma                         0.5424          0.5124    -0.0300
human_rating_alpha                      0.1426          0.1352    -0.0074
human_rating_beta                       

In [29]:
df_soc["SOC_per_CPS"] = df_soc.groupby("CPS Code")["CPS Code"].transform("size")

is_mapped = df_soc["SOC_per_CPS"] <= 1

mapped = df_soc[is_mapped]

# Drop all other name columns
cols_to_drop = CODE_COLS - {'CPS Code', 'CPS Title'}
mapped = mapped.drop(
    columns=cols_to_drop, errors="ignore"
)
mapped['agg_level_SOC'] = False
mapped['agg_level_OEWS'] = False

to_map = df_soc[~is_mapped]

print(mapped.columns)

Index(['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma',
       'CPS Code', 'CPS Title', 'dv_rating_alpha_std_ONET',
       'dv_rating_beta_std_ONET', 'dv_rating_gamma_std_ONET',
       'human_rating_alpha_std_ONET', 'human_rating_beta_std_ONET',
       'human_rating_gamma_std_ONET', 'agg_level_ONET', 'SOC_per_CPS',
       'agg_level_SOC', 'agg_level_OEWS'],
      dtype='str')


In [30]:
# 39-1013 had to be merged to one SOC code from 2 in the 2010->2018 shift
# The merged cell appears as a duplicate
# It maps 1:1 to OEWS so dropping
######### FIX FIX FIX: some groups dont add to 100 because of OPENAI dropped units
oews_weights_clean = oews_weights.drop_duplicates(subset='2018 SOC code', keep='first')

to_map_with_weights = to_map.merge(oews_weights_clean[['2018 SOC code', '2018 weight']],
                                   left_on="SOC Code",
                                   right_on="2018 SOC code",
                                   how="left",
                                   indicator=True)

assert (to_map_with_weights['_merge'] == 'both').sum() == 13, "Incorrect number merged at this level"

to_map_with_weights = to_map_with_weights.drop(columns=["2018 SOC code", "_merge"])

In [31]:
# Coerce because of artifacts from the html table such as dashes as placeholder
to_map_with_weights['2018 weight'] = pd.to_numeric(
    to_map_with_weights['2018 weight'], 
    errors='coerce'
)

# Give every 1:1 a weight of 1, give every non listed collapse a simgple average
# Average is BLS fallback for when data is not available
group_size = to_map_with_weights.groupby('OEWS Code')['SOC Code'].transform('size')
to_map_with_weights['2018 weight'] = to_map_with_weights['2018 weight'].fillna(1 / group_size)

In [32]:
# Collapse scores down to the OEWS level
to_map_with_weights['agg_level_SOC'] = to_map_with_weights.duplicated(subset=['OEWS Code'], keep=False)

# Create standard deviation for merged codes
std_df_soc =  to_map_with_weights.groupby('OEWS Code')[INDEX_COLS].std().reset_index()
std_df_soc = std_df_soc.rename(columns={col: f'{col}_std_SOC' for col in INDEX_COLS})

# Merge in standard deviation and indicator of aggregation at this level
df_soc_with_stats = to_map_with_weights.merge(std_df_soc, on='OEWS Code', how='left')

collapsed_rows_soc = df_soc_with_stats[df_soc_with_stats['agg_level_SOC'] == True]
collapsed_rows_soc.to_csv('../temp/collapsed_at_SOC.csv', index=False)

code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title'}

for col in INDEX_COLS:
    df_soc_with_stats[col] = pd.to_numeric(df_soc_with_stats[col], errors='coerce')
    df_soc_with_stats[f'{col}_weighted'] = df_soc_with_stats[col] * df_soc_with_stats['2018 weight']


# Average the index values and keep all other columns
agg_dict = {f'{col}_weighted': 'sum' for col in INDEX_COLS}
agg_dict.update({col: 'first' for col in code_cols_for_agg})
agg_dict.update({f'{col}_std_ONET': 'first' for col in INDEX_COLS})
agg_dict.update({f'{col}_std_SOC': 'first' for col in INDEX_COLS})
agg_dict.update({'agg_level_ONET': 'first'})
agg_dict.update({'agg_level_SOC': 'first'})
agg_dict.update({'SOC_per_CPS': 'first'})

df_oews = df_soc_with_stats.groupby('OEWS Code', as_index=False).agg(agg_dict)

assert len(df_oews) <= df['OEWS Code'].nunique()
assert df_oews['OEWS Code'].is_unique, "duplicate OEWS Codes survived the groupby"

df_oews = df_oews.rename(columns={f'{col}_weighted': f'{col}' for col in INDEX_COLS})

In [33]:
print(oews.columns)

Index(['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE',
       'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP',
       'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
       'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN',
       'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75',
       'A_PCT90', 'ANNUAL', 'HOURLY'],
      dtype='str')


In [34]:
print(df_oews.columns)

Index(['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma',
       'manual_soc_to_cps', 'CPS Code', 'OEWS Code', 'CPS Title', 'OEWS Title',
       'dv_rating_alpha_std_ONET', 'dv_rating_beta_std_ONET',
       'dv_rating_gamma_std_ONET', 'human_rating_alpha_std_ONET',
       'human_rating_beta_std_ONET', 'human_rating_gamma_std_ONET',
       'dv_rating_alpha_std_SOC', 'dv_rating_beta_std_SOC',
       'dv_rating_gamma_std_SOC', 'human_rating_alpha_std_SOC',
       'human_rating_beta_std_SOC', 'human_rating_gamma_std_SOC',
       'agg_level_ONET', 'agg_level_SOC', 'SOC_per_CPS'],
      dtype='str')


In [35]:
# Now merge in OEWS total employment before final weighted merge down to CPS
df_oews_with_emp = df_oews.merge(oews[['OCC_CODE', 'TOT_EMP']],
                                 left_on='OEWS Code',
                                 right_on='OCC_CODE',
                                 how='left',
                                 indicator=True)

print(df_oews_with_emp['_merge'].value_counts())

assert (df_oews_with_emp['_merge'] == 'left_only').sum() == 0, "Some OEWS codes did not get their employment merged"

df_oews_with_emp = df_oews_with_emp.drop(columns=['OCC_CODE', '_merge'])

_merge
both          420
left_only       0
right_only      0
Name: count, dtype: int64


In [36]:
# Calculate employment shares from total employment
df_oews_with_emp['TOT_EMP'] = pd.to_numeric(df_oews_with_emp['TOT_EMP'])
df_oews_with_emp['emp_share'] = df_oews_with_emp.groupby('CPS Code')['TOT_EMP'].transform(lambda x: x / x.sum())

In [37]:
# Collapse scores down to the CPS level
df_oews_with_emp['agg_level_OEWS'] = df_oews_with_emp.duplicated(subset=['CPS Code'], keep=False)

# Create standard deviation for merged codes
std_df_oews =  df_oews_with_emp.groupby('CPS Code')[INDEX_COLS].std().reset_index()
std_df_oews = std_df_oews.rename(columns={col: f'{col}_std_CPS' for col in INDEX_COLS})

# Merge in standard deviation and indicator of aggregation at this level
df_oews_with_stats = df_oews_with_emp.merge(std_df_oews, on='CPS Code', how='left')

collapsed_rows_cps = df_oews_with_stats[df_oews_with_stats['agg_level_OEWS'] == True]
collapsed_rows_cps.to_csv('../temp/collapsed_at_cps.csv', index=False)

for col in INDEX_COLS:
     df_oews_with_stats[f'{col}_std_CPS'] = df_oews_with_stats[f'{col}_std_CPS'].fillna(0) 

code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title', 'OEWS Code', 'OEWS Title'}

for col in INDEX_COLS:
    df_oews_with_stats[col] = pd.to_numeric(df_oews_with_stats[col], errors='coerce')
    df_oews_with_stats[f'{col}_weighted'] = df_oews_with_stats[col] * df_oews_with_stats['emp_share']

# Average the index values and keep all other columns
agg_dict = {f'{col}_weighted': 'sum' for col in INDEX_COLS}
agg_dict.update({col: 'first' for col in code_cols_for_agg})
agg_dict.update({f'{col}_std_ONET': 'first' for col in INDEX_COLS})
agg_dict.update({f'{col}_std_SOC': 'first' for col in INDEX_COLS})
agg_dict.update({f'{col}_std_CPS': 'first' for col in INDEX_COLS})
agg_dict.update({'agg_level_ONET': 'first'})
agg_dict.update({'agg_level_SOC': 'first'})
agg_dict.update({'agg_level_OEWS': 'first'})
agg_dict.update({'SOC_per_CPS': 'first'})

df_cps = df_oews_with_stats.groupby('CPS Code', as_index=False).agg(agg_dict)

assert len(df_cps) <= df_cps['CPS Code'].nunique()
assert df_cps['CPS Code'].is_unique, "duplicate CPS Codes survived the groupby"

df_cps = df_cps.rename(columns={f'{col}_weighted': f'{col}' for col in INDEX_COLS})

In [38]:
print(df_cps.columns)


Index(['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma',
       'CPS Title', 'manual_soc_to_cps', 'CPS Code',
       'dv_rating_alpha_std_ONET', 'dv_rating_beta_std_ONET',
       'dv_rating_gamma_std_ONET', 'human_rating_alpha_std_ONET',
       'human_rating_beta_std_ONET', 'human_rating_gamma_std_ONET',
       'dv_rating_alpha_std_SOC', 'dv_rating_beta_std_SOC',
       'dv_rating_gamma_std_SOC', 'human_rating_alpha_std_SOC',
       'human_rating_beta_std_SOC', 'human_rating_gamma_std_SOC',
       'dv_rating_alpha_std_CPS', 'dv_rating_beta_std_CPS',
       'dv_rating_gamma_std_CPS', 'human_rating_alpha_std_CPS',
       'human_rating_beta_std_CPS', 'human_rating_gamma_std_CPS',
       'agg_level_ONET', 'agg_level_SOC', 'agg_level_OEWS', 'SOC_per_CPS'],
      dtype='str')


In [39]:
print(mapped.columns)

Index(['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma',
       'CPS Code', 'CPS Title', 'dv_rating_alpha_std_ONET',
       'dv_rating_beta_std_ONET', 'dv_rating_gamma_std_ONET',
       'human_rating_alpha_std_ONET', 'human_rating_beta_std_ONET',
       'human_rating_gamma_std_ONET', 'agg_level_ONET', 'SOC_per_CPS',
       'agg_level_SOC', 'agg_level_OEWS'],
      dtype='str')


In [40]:
df_cps = pd.concat([df_cps, mapped])

In [41]:
# Calculate final std
std_df_cps =  df.groupby('CPS Code')[INDEX_COLS].std().reset_index()
std_df_cps = std_df_cps.rename(columns={col: f'{col}_std_pooled' for col in INDEX_COLS})

# Merge in standard deviation and indicator of aggregation at this level
df_cps_with_std = df_cps.merge(std_df_cps, on='CPS Code', how='left')


for col in INDEX_COLS:
    df_cps_with_std[f'{col}_std_pooled'] = df_cps_with_std[f'{col}_std_pooled'].fillna(0)

In [42]:
df_cps_with_std = df_cps_with_std.sort_values(by='CPS Code').reset_index(drop=True)
df_cps_with_std.to_csv('../data/full_exposure.csv', index=False)

In [43]:
df_final = df_cps_with_std[['CPS Code', 'CPS Title', 'dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma','dv_rating_alpha_std_pooled', 'dv_rating_beta_std_pooled',
       'dv_rating_gamma_std_pooled', 'human_rating_alpha_std_pooled',
       'human_rating_beta_std_pooled', 'human_rating_gamma_std_pooled', 'SOC_per_CPS', 'agg_level_ONET', 'agg_level_SOC', 'agg_level_OEWS']]
df_final.to_csv('../data/full_exposure_reduced.csv', index=False)

In [44]:
df_small = df_final[['CPS Code', 'CPS Title', 'dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma']]
df_small.to_csv('../data/final_exposure.csv', index=False)